In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import struct

I0000 00:00:1781095537.066407  125399 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781095537.132503  125399 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781095538.537729  125399 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# PyTorch: transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
# uint8(0~255) -> float32(0.0~1.0) -> float32(-1.0~1.0)
def transform(img):
    img = img.astype(np.float32) / 255.0
    return (img - 0.5) / 0.5

In [3]:
class CustomDataset:
    def __init__(self, images, labels=None, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]

        if self.transform:
            img = self.transform(img)

        if self.labels is not None:
            return img, int(self.labels[idx])

        return img

In [4]:
with open('../datasets/mnist/train-images.idx3-ubyte', 'rb') as f:
    _, n, rows, cols = struct.unpack('>IIII', f.read(16))
    images = np.fromfile(f, dtype=np.uint8).reshape(n, rows, cols, 1)  # NHWC: (n, H, W, C)

with open('../datasets/mnist/train-labels.idx1-ubyte', 'rb') as f:
    struct.unpack('>II', f.read(8))
    labels = np.fromfile(f, dtype=np.uint8)

with open('../datasets/mnist/t10k-images.idx3-ubyte', 'rb') as f:
    _, n, rows, cols = struct.unpack('>IIII', f.read(16))
    test_images = np.fromfile(f, dtype=np.uint8).reshape(n, rows, cols, 1)  # NHWC

with open('../datasets/mnist/t10k-labels.idx1-ubyte', 'rb') as f:
    struct.unpack('>II', f.read(8))
    test_labels = np.fromfile(f, dtype=np.uint8)

images      = images.astype(np.float32)      / 255.0
test_images = test_images.astype(np.float32) / 255.0

In [5]:
train_dataset = CustomDataset(images, labels)
test_dataset  = CustomDataset(test_images, test_labels)

# tf.data.Dataset: PyTorch의 DataLoader 역할
train_loader = tf.data.Dataset.from_tensor_slices((images, labels)).shuffle(len(images)).batch(64).prefetch(tf.data.AUTOTUNE)
test_loader  = tf.data.Dataset.from_tensor_slices((test_images, test_labels)).batch(64).prefetch(tf.data.AUTOTUNE)

I0000 00:00:1781095540.855556  125399 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:1e.0, compute capability: 7.5
